<a href="https://colab.research.google.com/github/sairas2124/Gradient_descent-/blob/main/Handeling_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import make_classification
import torch

In [ ]:
X,y = make_classification(
   n_samples = 10,
   n_features = 2,
   n_informative = 2,
   n_redundant = 0,
   n_classes=2,
   random_state=42

)

In [ ]:
X.shape

(10, 2)

In [ ]:
#convert dataset in pytorch tensor
X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y,dtype=torch.long)

In [ ]:
X

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])

In [ ]:
from torch.utils.data import Dataset,DataLoader

In [ ]:
class CustomDataset(Dataset):
   def __init__(self,features,labels):
      self.features = features
      self.labels = labels

   def __len__(self):
      return self.features.shape[0]

   def __getitem__(self,idx):
      return self.features[idx],self.labels[idx]

In [ ]:
dataset = CustomDataset(X,y)

In [ ]:
dataloader = DataLoader(dataset, batch_size= 20, shuffle=True)

In [ ]:
for batch_features, batch_labels in dataloader:

  print(batch_features)
  print(batch_labels)
  print("-"*50)

tensor([[-0.9382, -0.5430],
        [-0.7206, -0.9606],
        [ 1.7774,  1.5116],
        [ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-1.9629, -0.9923],
        [ 1.8997,  0.8344],
        [-2.8954,  1.9769],
        [ 1.7273, -1.1858],
        [-0.5872, -1.9717]])
tensor([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])
--------------------------------------------------


In [18]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [19]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [20]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [22]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [23]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [35]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [25]:
class CustomDataset(Dataset):
   def __init__(self,features,labels):
      self.features = features
      self.labels = labels

   def __len__(self):
      return self.features.shape[0]

   def __getitem__(self,idx):
      return self.features[idx],self.labels[idx]

In [26]:
train_dataset = CustomDataset(X_train_tensor,y_train_tensor)
test_dataset = CustomDataset(X_test_tensor,y_test_tensor)

In [27]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size = 32,shuffle=True)

In [28]:
import torch.nn as nn
class MySimple(nn.Module):
  def __init__(self,num_features):
     super().__init__()
     self.linear = nn.Linear(num_features, 1)
     self.sigmoid = nn.Sigmoid()


  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out


In [29]:
learning_rate = 0.1
epochs = 25

In [30]:
# create model
model = MySimple(X_train_tensor.shape[1])

#defining optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

#automated loss_function
loss_function = nn.BCELoss()

In [33]:
for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:
       # forward pass
       y_pred = model(X_train_tensor.float())

       # loss calculate
       loss = loss_function(y_pred, y_train_tensor.view(-1,1).float())

       # backward pass
       loss.backward()

       # parameters update
       optimizer.step()
       optimizer.zero_grad()

      # print loss in each epoch
       print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.4425611197948456
Epoch: 1, Loss: 0.3711601495742798
Epoch: 1, Loss: 0.3259243071079254
Epoch: 1, Loss: 0.294513463973999
Epoch: 1, Loss: 0.27128180861473083
Epoch: 1, Loss: 0.25329598784446716
Epoch: 1, Loss: 0.23888342082500458
Epoch: 1, Loss: 0.2270202785730362
Epoch: 1, Loss: 0.21704424917697906
Epoch: 1, Loss: 0.2085074484348297
Epoch: 1, Loss: 0.20109570026397705
Epoch: 1, Loss: 0.19458186626434326
Epoch: 1, Loss: 0.18879744410514832
Epoch: 1, Loss: 0.18361468613147736
Epoch: 1, Loss: 0.17893482744693756
Epoch: 2, Loss: 0.17468026280403137
Epoch: 2, Loss: 0.17078909277915955
Epoch: 2, Loss: 0.16721130907535553
Epoch: 2, Loss: 0.1639060229063034
Epoch: 2, Loss: 0.16083940863609314
Epoch: 2, Loss: 0.15798333287239075
Epoch: 2, Loss: 0.15531402826309204
Epoch: 2, Loss: 0.15281140804290771
Epoch: 2, Loss: 0.15045830607414246
Epoch: 2, Loss: 0.1482398957014084
Epoch: 2, Loss: 0.14614342153072357
Epoch: 2, Loss: 0.14415772259235382
Epoch: 2, Loss: 0.14227308332920074
E

In [37]:
# model evaluation
model.eval()
accuracy_list=[]

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    y_pred = model(batch_features.float())
    y_pred = (y_pred > 0.8).float()

    accuracy = (y_pred == batch_labels.view(-1,1).float()).float().mean().item()
    print(f'Accuracy: {accuracy}')
    accuracy_list.append(accuracy)

overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Overall Accuracy: {overall_accuracy}')

Accuracy: 0.875
Accuracy: 0.96875
Accuracy: 0.9375
Accuracy: 0.8333333134651184
Overall Accuracy: 0.9036458283662796
